<a href="https://colab.research.google.com/github/Aswanth0704/gpu-programming-cpp/blob/main/7_3_Excercise_NVTX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
if os.getenv("COLAB_RELEASE_TAG"): # If running in Google Colab:
  !mkdir -p Sources
  !wget https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/02.02-Asynchrony/Sources/ach.h -nv -O Sources/ach.h
  !wget https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/02.02-Asynchrony/Sources/nvtx3.hpp -nv -O Sources/nvtx3.hpp
  !sudo apt-key adv --fetch-keys https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64/7fa2af80.pub > /dev/null 2>&1
  !sudo add-apt-repository -y "deb https://developer.download.nvidia.com/devtools/repos/ubuntu$(source /etc/lsb-release; echo "$DISTRIB_RELEASE" | tr -d .)/$(dpkg --print-architecture)/ /" > /dev/null 2>&1
  !sudo apt install -y nsight-systems > /dev/null 2>&1

2026-09-27 22:32:16 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/02.02-Asynchrony/Sources/ach.h [5329/5329] -> "Sources/ach.h" [1]
2026-09-27 22:32:16 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/02.02-Asynchrony/Sources/nvtx3.hpp [105839/105839] -> "Sources/nvtx3.hpp" [1]


In [ ]:
%%writefile Sources/nvtx.cpp
#include "ach.h"

void simulate(int width, int height, const thrust::device_vector<float> &in,
              thrust::device_vector<float> &out)
{
  cuda::std::mdspan temp_in(thrust::raw_pointer_cast(in.data()), height, width);
  cub::DeviceTransform::Transform(
      thrust::make_counting_iterator(0), out.begin(), width * height,
      [=] __host__ __device__(int id) { return ach::compute(id, temp_in); });
}

int main()
{
  int height = 2048;
  int width = 8192;

  thrust::device_vector<float> d_prev = ach::init(height, width);
  thrust::device_vector<float> d_next(height * width);
  thrust::host_vector<float> h_prev(height * width);

  const int compute_steps = 750;
  const int write_steps = 3;
  for (int write_step = 0; write_step < write_steps; write_step++)
  {
    nvtx3::scoped_range r{std::string("write step ") + std::to_string(write_step)};

    {
      nvtx3::scoped_range r{"copy"};
      thrust::copy(d_prev.begin(), d_prev.end(), h_prev.begin());
    }

    {

      nvtx3::scoped_range r{"compute"};
      for (int compute_step = 0; compute_step < compute_steps; compute_step++)
      {
        simulate(width, height, d_prev, d_next);
        d_prev.swap(d_next);
      }
    }

    {

      nvtx3::scoped_range r{"write"};
      ach::store(write_step, height, width, h_prev);
    }

    {

      nvtx3::scoped_range r{"wait"};
      cudaDeviceSynchronize();
    }
  }
}

In [ ]:
!nvcc --extended-lambda -o /tmp/a.out Sources/nvtx.cpp -x cu -arch=native # build executable
!nsys profile --cuda-event-trace=false --force-overwrite true -o nvtx /tmp/a.out # run and profile executable